# Alunos
- **Matheus Peixoto Ribeiro Vieira - 22.1.4104**
- **Pedro Henrique Rabelo Leão de Oliveira - 22.1.4022**

# Objetivo do desafio
A partir de uma base de dados com partidas do campeonato brasileiro de futebol desde o ano 2012 até atualmente, o objetivo é predizer o resultado de um novo confronto, dizendo se a partida acabará em vitória de um dos times ou em empate.

# Mineração de Regras de Associação

In [1]:
import pandas as pd

import json
with open('data/BRA-modified.json', 'r') as f:
    data = json.load(f)

data = {id: name for name, id in zip(data.keys(), data.values())}
data

id_to_name = lambda id: data[id]

## Discretizando os valores numéricos

In [2]:
def freedman_diaconis_rule(data):
    iqr = data.quantile(0.75) - data.quantile(0.25)
    data = data.to_numpy()
    num_bins = int((data.max()-data.min()) / (2*iqr*len(data)**(-1/3)))
    return num_bins

In [3]:
# dataset sem o pre-processamento feito no arquivo 'pre_processing.ipynb'
original_dataset = pd.read_csv('data/BRA.csv')
original_dataset = original_dataset.drop(1891)

In [4]:
def discretiza_odds(valor_odd):
    """ 
    Discretiza os valores das odds, classificando em: 
     - 0 (time favorito da partida) quando a odd é menor que 2; 
     - 1 (equilibrado) quando a odd está no intervalo [2, 3[;
     - 2 (time improvável de vencer) quando a odd é maior ou igual a 3.
    """
    if valor_odd < 2:
        return 0
    elif valor_odd < 3:
        return 1
    else:
        return 2

df = pd.read_csv('BRA-pre-processed.csv')
categorical_data = ["Home", "Away", "Res_D", "Res_H", "WLF_H", "DLF_H", "LLF_H", "WLF_A", "DLF_A", "LLF_A"]
numerical_columns = list(set(df.columns.to_list()) - set(categorical_data))

for col in numerical_columns:
    if col in ["PSCH", "PSCD", "PSCA", "MaxCH", "MaxCD", "MaxCA", "AvgCA", "AvgCH", "AvgCD"]:
        df[col] = original_dataset[col].apply(discretiza_odds)

    else:
        num_bins = freedman_diaconis_rule(df[col])
        value_bins = pd.cut(df[col], num_bins, labels=range(0, num_bins))
        df[col] = value_bins


In [5]:
df

,Home,Away,PSCH,PSCD,PSCA,MaxCH,MaxCD,MaxCA,AvgCH,AvgCD,...,GA_A,GD_A,WLF_H,DLF_H,LLF_H,WLF_A,DLF_A,LLF_A,Res_D,Res_H
0,27,30,0.0,2.0,2.0,0.0,2.0,2.0,0.0,2.0,...,0,47,0.0,0.0,0.0,0.0,0.0,0.0,1,0
1,34,17,1.0,2.0,1.0,1.0,2.0,1.0,1.0,2.0,...,0,47,0.0,0.0,0.0,0.0,0.0,0.0,1,0
2,16,26,0.0,2.0,2.0,0.0,2.0,2.0,0.0,2.0,...,0,47,0.0,0.0,0.0,0.0,0.0,0.0,0,1
3,6,33,1.0,2.0,2.0,1.0,2.0,2.0,1.0,2.0,...,0,47,0.0,0.0,0.0,0.0,0.0,0.0,0,1
4,11,18,0.0,2.0,2.0,0.0,2.0,2.0,0.0,2.0,...,0,47,0.0,0.0,0.0,0.0,0.0,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5050,36,14,1.0,2.0,2.0,1.0,2.0,2.0,1.0,2.0,...,2,55,0.2,0.2,0.6,0.8,0.2,0.0,1,0
5051,19,32,2.0,2.0,1.0,2.0,2.0,1.0,2.0,2.0,...,4,44,0.2,0.2,0.6,0.2,0.2,0.6,0,0
5052,21,11,1.0,2.0,2.0,1.0,2.0,2.0,1.0,2.0,...,5,46,0.6,0.2,0.2,0.4,0.4,0.2,1,0
5053,3,22,1.0,1.0,2.0,1.0,2.0,2.0,1.0,1.0,...,5,44,0.6,0.4,0.0,0.0,0.4,0.6,0,1


## Regras de Associação

In [76]:
transactions = []

for index_row, row in df.iterrows():
    res = ('H' if (row['Res_H'] == 1) else 
           'D' if (row['Res_D'] == 1) else 
           'A')

    num_goals = original_dataset.iloc[index_row]['HG'] + original_dataset.iloc[index_row]['AG']
    total_goals = "Mais_2.5_Gols" if (num_goals >= 3) else "Menos_2.5_Gols"

    odd_media_casa = ('Odd_Casa_Baixa' if (row['AvgCH'] == 0) else 
                      'Odd_Casa_Média' if (row['AvgCH'] == 1) else 
                      'Odd_Casa_Alta')
    
    odd_media_empate = ('Odd_Empate_Baixa' if (row['AvgCD'] == 0) else 
                        'Odd_Empate_Média' if (row['AvgCD'] == 1) else 
                        'Odd_Empate_Alta')
    
    odd_media_visitante = ('Odd_Visitante_Baixa' if (row['AvgCA'] == 0) else 
                           'Odd_Visitante_Média' if (row['AvgCA'] == 1) else 
                           'Odd_Visitante_Alta')

    itens = [
        f"Casa_{int(row['Home'])}",
        f"Visitante_{int(row['Away'])}",
        total_goals, # total de gols na partida
        f"Vitórias_Casa_{row['WLF_H']}",     # Vítorias do time da casa nas 5 partidas anteriores
        f"Vitórias_Visitante_{row['WLF_A']}", # Vítorias do time visitante nas 5 partidas anteriores
        f"Resultado_{res}",
        odd_media_casa,
        odd_media_empate,
        odd_media_visitante
    ]

    transactions.append(itens)

for t in transactions[:5]:
    print(t)

['Casa_27', 'Visitante_30', 'Menos_2.5_Gols', 'Vitórias_Casa_0.0', 'Vitórias_Visitante_0.0', 'Resultado_D', 'Odd_Casa_Baixa', 'Odd_Empate_Alta', 'Odd_Visitante_Alta']
['Casa_34', 'Visitante_17', 'Menos_2.5_Gols', 'Vitórias_Casa_0.0', 'Vitórias_Visitante_0.0', 'Resultado_D', 'Odd_Casa_Média', 'Odd_Empate_Alta', 'Odd_Visitante_Média']
['Casa_16', 'Visitante_26', 'Mais_2.5_Gols', 'Vitórias_Casa_0.0', 'Vitórias_Visitante_0.0', 'Resultado_H', 'Odd_Casa_Baixa', 'Odd_Empate_Alta', 'Odd_Visitante_Alta']
['Casa_6', 'Visitante_33', 'Mais_2.5_Gols', 'Vitórias_Casa_0.0', 'Vitórias_Visitante_0.0', 'Resultado_H', 'Odd_Casa_Média', 'Odd_Empate_Alta', 'Odd_Visitante_Média']
['Casa_11', 'Visitante_18', 'Menos_2.5_Gols', 'Vitórias_Casa_0.0', 'Vitórias_Visitante_0.0', 'Resultado_A', 'Odd_Casa_Baixa', 'Odd_Empate_Alta', 'Odd_Visitante_Alta']


### Executando o apriori

A biblioteca mlxtend.frequent_patterns necessita que todos os dados estejam em formato one-hot encoded. Todavia trabalhamos com valores númericos contínuos. Dessa forma, transformar a discretização de todos os atributos em one-hot elevaria muito o custo de memória. Por isso o seu uso não será escolhido e utilizaremos o apyori.

In [77]:
from apyori import apriori

In [78]:
association_rules = apriori(transactions, min_support=0.2, min_confidence=0.6, min_lift=1.2) 
association_results = list(association_rules)

for r in association_results:
    itens = [x for x in r.items]
    print(f"Regra: {itens}")
    print(f"Suporte: {r.support:.3f}")
    for o in r.ordered_statistics:
        print(f"  {list(o.items_base)} => {list(o.items_add)}")
        print(f"  Confiança: {o.confidence:.3f}")
        print(f"  Lift: {o.lift:.3f}")
    print("-" * 50)

Regra: ['Menos_2.5_Gols', 'Resultado_D']
Suporte: 0.216
  ['Resultado_D'] => ['Menos_2.5_Gols']
  Confiança: 0.802
  Lift: 1.409
--------------------------------------------------
Regra: ['Odd_Casa_Baixa', 'Odd_Visitante_Alta']
Suporte: 0.475
  ['Odd_Casa_Baixa'] => ['Odd_Visitante_Alta']
  Confiança: 1.000
  Lift: 1.414
  ['Odd_Visitante_Alta'] => ['Odd_Casa_Baixa']
  Confiança: 0.671
  Lift: 1.414
--------------------------------------------------
Regra: ['Mais_2.5_Gols', 'Odd_Casa_Baixa', 'Odd_Visitante_Alta']
Suporte: 0.210
  ['Mais_2.5_Gols', 'Odd_Casa_Baixa'] => ['Odd_Visitante_Alta']
  Confiança: 1.000
  Lift: 1.414
  ['Mais_2.5_Gols', 'Odd_Visitante_Alta'] => ['Odd_Casa_Baixa']
  Confiança: 0.687
  Lift: 1.447
--------------------------------------------------
Regra: ['Menos_2.5_Gols', 'Odd_Casa_Baixa', 'Odd_Visitante_Alta']
Suporte: 0.264
  ['Menos_2.5_Gols', 'Odd_Casa_Baixa'] => ['Odd_Visitante_Alta']
  Confiança: 1.000
  Lift: 1.414
  ['Menos_2.5_Gols', 'Odd_Visitante_Alta']

(Suporte Mínimo 0.3, confiança mínima 0.2 e lift mínimo 0.8)
- O item "Odd_Visitante_Baixa" não atendeu ao suporte mínimo, o que confirma a questão do favoritismo presente para os times que jogam em casa no brasileirão vista nas análises feitas por nós na primeira fase do trabalho. 
- Foi possível observar também a presença de odds altas para empates nas partidas, presente em 91,8% das partidas. 
- Além disso, 57% das partidas tiveram 2 ou menos gols, o que confirma o quão equilibrado o campeonato brasileiro é. 
- Uma regra encontrada foi o time da casa vencendo 48,4% das partidas, valor esses encontrado por nós nas análises feitas durante a primeira fase do trabalho e confirmando mais uma vez o favoritismo de quem joga em casa, ao lado de sua torcida.
- Em relação a regras com antecedentes, foi possível observar que toda vez que a odd média para o time da casa era baixa a odd média para o time visitante era alta, tendo assim uma confiança de 100%, o que é esperado no momento em que há o favoritismo para o time da casa. 
- Por fim, em 51,1% das vezes que temos uma odd média alta para o visitante, temos o time da casa vencendo a partida.

(Suporte Mínimo 0.2, confiança mínima de 0.6 e lift mínimo de 1.2)
- Foi possível observar que quando temos um empate, em 80,2% das vezes, acontecem 2 ou menos gols na partida, e o lift de 1,409 traz uma relação forte para o antecedente e o consequente. Isso condiz com o equilíbrio que se espera de uma partida que termina empatada, onde, nesses casos, acontecem menos gols.
- Aqui, foi possível confirmar também a questão da presença da odd média alta para o visitante em 100% das vezes que temos a odd média baixa para o time da casa.
- Além disso, no momento em que temos o resultado da partida com o time da casa vencendo e a odd média para esse time baixa, em 100% das vezes as odds médias para o visitante e para empate são altas, confirmando o favoritismo do time da casa.
- É possível perceber que uma das regras é composta pela odd média baixa para o time da casa, time da casa vencendo e odds médias altas para o time visitante e para o empate, o que indica que esses atributos estão relacionados normalmente.

(Suporte Mínimo 0.3, confiança mínima de 0.3 e lift mínimo de 0.9)
- Foi possível observar que, sempre que temos a presença da odd média baixa para o time da casa, também temos a presença das odds médias altas para o empate e para o visitante, com confiança de 100%, o que reforça o favoritismo do time da casa e a baixa expectativa de equilíbrio no confronto.
- A recíproca também ocorre, onde odds altas para o visitante ou para o empate implicam na odd baixa para o time da casa, com confianças de 67,1% e 51,7%, respectivamente, mostrando consistência na forma como as odds refletem o desequilíbrio entre as equipes.
- Em relação ao número de gols, foi identificado que quando ocorrem mais de 2.5 gols na partida, é comum a presença de odds altas para o empate e para o visitante, com confianças de 91,4% e 71,2%, respectivamente. Já quando ocorrem menos de 2.5 gols, há uma regra frequente envolvendo também odds altas para empate e visitante, com confiança de 67,3%.
- Em relação aos resultados, observou-se que os desfechos desfavoráveis ao time da casa não são frequentes com os hiperparâmetros utilizados, o que reforça o padrão de favoritismo do mandante.
- Também foi possível verificar que, quando temos odds altas tanto para o empate quanto para o visitante, há elevada confiança para a vitória do time da casa, com valores de confiança que variam, sendo o mínimo de 34,5%, mesmo sem antecedentes adicionais, reforçando a força do mandante nas partidas.

# Padrões sequenciais

In [6]:
def cria_sequencias_de_resultados(year):
    """
        Sequência de resultados de cada um dos times ao longo da temporada
        Itemset é formado por
            - Resultado desse time na partida: W (Win), D (Draw), L (Lose)
            - Quantos gols fez, mais ou menos do que 2.5
            - Quantos gols levou, mais ou menos do que 2.5
            - Odd média para o time vencer
            - Número de vitórias do time nas últimas 5 partidas
    """
    sequencia_de_resultados = dict()

    inicio_temporada = (original_dataset['Season'] == year).idxmax()
    if year > 2016: inicio_temporada -= 1

    final_temporada = inicio_temporada + 380 if (inicio_temporada + 380 <= original_dataset.index[-1]) else original_dataset.index[-1]

    # Sequencia de resultados de um time especifico
    for index_row, row in df.iloc[inicio_temporada:final_temporada].iterrows():
        home, away = row["Home"], row["Away"]

        res = (
            'H' if (row['Res_H'] == 1) else 
            'D' if (row['Res_D'] == 1) else 
            'A'
        )

        if res == "H": 
            h, a = "W", "L"
            
        elif res == "D": 
            h = a = "D"

        else: 
            h, a = "L", "W"
        
        hg = "Mais_2.5_Gols" if (original_dataset.iloc[index_row]['HG'] >= 3) else "Menos_2.5_Gols"
        ag = "Mais_2.5_Gols" if (original_dataset.iloc[index_row]['AG'] >= 3) else "Menos_2.5_Gols"

        
        odd_media_casa = (
            'Odd_Baixa' if (row['AvgCH'] == 0) else 
            'Odd_Média' if (row['AvgCH'] == 1) else 
            'Odd_Alta'
        )
        
        odd_media_visitante = (
            'Odd_Baixa' if (row['AvgCA'] == 0) else 
            'Odd_Média' if (row['AvgCA'] == 1) else 
            'Odd_Alta'
        )

        # vitorias_ultimas_5_partidas_casa = f"{int(row['WLF_H'])}_vitórias_nas_ultimas_5"
        # vitorias_ultimas_5_partidas_visitante = f"{int(row['WLF_A'])}_vitórias_nas_ultimas_5"

        if home in sequencia_de_resultados:
            sequencia_de_resultados[home].append((h, f"Fez_{hg}", f"Levou_{ag}", odd_media_casa))
        else:
            sequencia_de_resultados[home] = [(h, f"Fez_{hg}", f"Levou_{ag}", odd_media_casa)]

        if away in sequencia_de_resultados:
            sequencia_de_resultados[away].append((a, f"Fez_{ag}", f"Levou_{hg}", odd_media_visitante))
        else:
            sequencia_de_resultados[away] = [(a, f"Fez_{ag}", f"Levou_{hg}", odd_media_visitante)]

    return list(sequencia_de_resultados.values())

In [7]:
print(cria_sequencias_de_resultados(year=2022))

[[('D', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Baixa'), ('W', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Alta'), ('L', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Alta'), ('L', 'Fez_Menos_2.5_Gols', 'Levou_Mais_2.5_Gols', 'Odd_Alta'), ('D', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Alta'), ('W', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Média'), ('W', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Alta'), ('L', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Média'), ('L', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Alta'), ('W', 'Fez_Mais_2.5_Gols', 'Levou_Mais_2.5_Gols', 'Odd_Média'), ('L', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Alta'), ('D', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Média'), ('W', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Baixa'), ('W', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Baixa'), ('W', 'Fez_Mais_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Baixa'), ('W', 'Fez_Menos_2.

## Sequência de resultados dos times ao longo das temporadas

In [15]:
from prefixspan import PrefixSpan

def busca_padroes_sequencia_uma_temporada(year, percentualSup=0.9):
    lista_de_sequencias = cria_sequencias_de_resultados(year)
    
    ps = PrefixSpan(lista_de_sequencias)
    minsup = int(percentualSup * len(lista_de_sequencias))
    padroes = ps.frequent(minsup=minsup)

    if len(padroes) > 0:
        padroes = sorted(padroes, key= lambda x: (-x[0], len(x[1])))
        for suporte, padrao in padroes:
            print(f"Suporte: {suporte}  \n\tPadrão: {padrao}")
            print("-"*200)

In [164]:
busca_padroes_sequencia_uma_temporada(year=2021)

Suporte: 20  
	Padrão: [('L', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Alta')]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Suporte: 20  
	Padrão: [('D', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Alta')]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Suporte: 20  
	Padrão: [('D', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Média')]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Suporte: 20  
	Padrão: [('L', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Média')]
---------------------------------------

Na temporada de 2021:
- 19 dos 20 times perderam um jogo já sendo o mais provável à derrota e em seguida venceram um jogo de forma inesperada. Essa expectativa pode ser confirmada pela odd alta em ambas as partidas.

- 18 dos 20 times chegaram a empatar mais de 1 vez quando tinham a odd ao seu favor em um valor mediano. Isso porque ambas as partidas foram contra adversários de nível similar ao seu e tanto a odd quanto o resultado confirmaram isso.

- 18 dos 20 times tiveram uma sequência de 3 derrotas tendo odds altas nas 3 partidas, ou seja, já sendo considerado o mais provável à derrota em todas.

- Em um suporte tão alto, de 90%, não foram encontrados padrões de sequência com times fazendo ou levando mais de 2.5 gols. Já o contrário foi encontrado, na maioria dos padrões, os times fazem e levam menos de 2.5 gols. Isso acontece devido ao equilíbrio do campeonato, onde pouco times possuem uma sequência de fazerem ou levaram 3 ou mais gols.

- Em um suporte tão alto, também não encontramos sequências de 4 vitórias por exemplo, já que, caso tenha acontecido, ocorreu com poucos times do campeonato.

In [165]:
busca_padroes_sequencia_uma_temporada(year=2022)

Suporte: 20  
	Padrão: [('D', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Baixa')]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Suporte: 20  
	Padrão: [('W', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Alta')]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Suporte: 20  
	Padrão: [('D', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Alta')]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Suporte: 20  
	Padrão: [('W', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Média')]
---------------------------------------

Na temporada de 2022:
- Diferentemente da temporada de 2021, na de 2022, mesmo com um suporte mínimo de 90%, houve o aparecimento de times que levaram mais de 2.5 gols em uma partida. Isso aconteceu com 19 dos 20 times, que levaram 3 ou mais gols, fez 2 ou menos e já sendo os mais prováveis à derrota para a partida de acordo com as odds. Portanto, percebemos que em 2022 os "apagões" defensivos foram mais comuns entre os times.

- Também encontramos um padrão que mostra o acontecimento de "zebras" nas partidas. Todos os times, inclusive o campeão, tiveram partidas em que eram favoritos e perderam. Isso aconteceu em partidas que o mesmo fez e levou menos que 2.5 gols, ou seja, partidas com poucos gols em que apenas um vacilo de um dos lados pode definí-la.

- Outro padrão que mostrou a dificuldade de sustentar uma boa fase foi o de times que ganharam uma partida e depois perderam. Esse padrão aconteceu com 18 times.

- 18 dos 20 times do campeonato tiveram uma sequência de vitórias em que eram considerados os mais prováveis à derrota. Nesses casos, podemos observar que a odd, após a partida que o time venceu, passa de alta para média.

- 18 dos 20 times tiveram uma sequência de derrota com odd alta, e depois duas vitórias, a primeira com odd alta e a segunda com odd baixa, mostrando que a sequência foi conquistando "respeito" para o time, impactando, de forma esperada, na sua odd.

- É possível perceber também que o empate foi bem presente na temporada, sendo uma sequência tripla de resultados para 18 dos 20 times.

In [166]:
busca_padroes_sequencia_uma_temporada(year=2023)

Suporte: 20  
	Padrão: [('D', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Média')]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Suporte: 20  
	Padrão: [('L', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Alta')]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Suporte: 20  
	Padrão: [('D', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Alta')]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Suporte: 20  
	Padrão: [('L', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Alta'), ('D', 'Fez_Menos_2.5_Gols', 'Levou_Menos

Na temporada 2023:

-  Todos os times tiveram uma sequência onde primeiro perderam um jogo e depois empataram o segundo, mostrando uma recuperação

- Todavia, não é provável que o time vença a próxima partida, uma vez que esse não é um padrão frequente. Logo, o mais comum é empatar ou perder. Também é possível que ele tenha perdido a partida anterior a essas duas.

- Em nenhuma das sequências obtidas que possui uma vitória, o time a fez com mais de 2.5 gols

- 18 times venceram duas partidas em sequência mesmo possuindo odds altas

- 18 times perderam uma partida quando suas odds de vitória eram baixas, mas isso não se repete mais de uma vez para ser feita uma sequência

In [167]:
busca_padroes_sequencia_uma_temporada(year=2024)

Suporte: 20  
	Padrão: [('L', 'Fez_Menos_2.5_Gols', 'Levou_Mais_2.5_Gols', 'Odd_Alta')]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Suporte: 20  
	Padrão: [('W', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Média')]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Suporte: 20  
	Padrão: [('L', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Alta')]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Suporte: 20  
	Padrão: [('W', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Alta')]
-----------------------------------------

Na temporada de 2024:
- Todos os times apresentaram uma sequência de Derrota, Derrota e Empate em partidas com poucos gols e odds altas para vencerem

- Com suporte 18, a maior sequência de vitórias são de duas partidas, fazendo menos de 2.5 gols, e todos os tipos de odds para as vitórias estão presentes

- Com suporte de 18, é notável uma vitória após duas derrotas seguidas, mas ela é feita com menos de 2.5 gols e com odds médias ou altas, mostrando que a conifança em relação à equipe não está alta

- Com suporte 18, um time que vence uma partida com odd média e perde a próxima com odd alta para vitória tende a ganhar a terceira, mas possuindo uma odd média ou alta para a vitória

- Com suporte de 19, um time que venceu uma partida tendo odds altas, tende a perder as duas seguintes, possuindo odds altas para vencer ambas

- Com suporte de 19, um time que perdeu duas em sequênica com odds altas tende a empatar a terceira possuindo odds baixas para vitória

In [168]:
busca_padroes_sequencia_uma_temporada(year=2025, percentualSup=0.6)

Suporte: 18  
	Padrão: [('L', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Alta')]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Suporte: 17  
	Padrão: [('D', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Alta')]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Suporte: 15  
	Padrão: [('W', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Média')]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Suporte: 14  
	Padrão: [('D', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Baixa')]
---------------------------------------

Na temporada de 2025, como ainda há poucos jogos, o suporte foi reduzido para 60% e as seguintes análises podem ser feitas:

- Não há sequências de duas ou mais vitórias que configurem uma padrão 
- Quando enfrentados com dois jogos seguidos com odds altas para vitória, é comum o time perder as duas 
- É mais comum empatar e depois perder do que perder e depois empatar

# Sequência de um único time

Cria a sequência de resultados de cada time ao longo das temporadas, onde cada temporada é uma sequência

O itemset consiste do resultado da partida em relação ao time analisado, a quantidade de gols feitos e levados e a odd para a vitória

In [8]:
from prefixspan import PrefixSpan

In [12]:
def sequencia_de_resultados_em_um_ano(year, team_id):
    """
        Sequência de resultados de cada um dos times ao longo da temporada
        Itemset é formado por
            - Resultado desse time na partida: W (Win), D (Draw), L (Lose)
            - Quantos gols fez, mais ou menos do que 2.5
            - Quantos gols levou, mais ou menos do que 2.5
            - Odd média para o time vencer
            - Número de vitórias do time nas últimas 5 partidas
    """
    sequencia_de_resultados = []

    inicio_temporada = (original_dataset['Season'] == year).idxmax()
    if year > 2016: inicio_temporada -= 1

    final_temporada = inicio_temporada + 380 if (inicio_temporada + 380 <= original_dataset.index[-1]) else original_dataset.index[-1]

    # Sequencia de resultados de um time especifico
    for index_row, row in df.iloc[inicio_temporada:final_temporada].iterrows():
        home, away = row["Home"], row["Away"]

        res = ('H' if (row['Res_H'] == 1) else 'D' if (row['Res_D'] == 1) else 'A')

        if res == "H": h, a = "W", "L"
        elif res == "D": h = a = "D"
        else: h, a = "L", "W"
        
        hg = "Mais_2.5_Gols" if (original_dataset.iloc[index_row]['HG'] >= 3) else "Menos_2.5_Gols"
        ag = "Mais_2.5_Gols" if (original_dataset.iloc[index_row]['AG'] >= 3) else "Menos_2.5_Gols"

        
        odd_media_casa = ('Odd_Baixa' if (row['AvgCH'] == 0) else 'Odd_Média' if (row['AvgCH'] == 1) else 'Odd_Alta')
        
        odd_media_visitante = ('Odd_Baixa' if (row['AvgCA'] == 0) else 'Odd_Média' if (row['AvgCA'] == 1) else 'Odd_Alta')


        if home == team_id:
            sequencia_de_resultados.append((h, f"Fez_{hg}", f"Levou_{ag}", odd_media_casa))
        elif away == team_id:
            sequencia_de_resultados.append((a, f"Fez_{ag}", f"Levou_{hg}", odd_media_visitante))


    return sequencia_de_resultados

def criar_sequencia_de_resultados_um_time(team_id):
    seasons = [sequencia_de_resultados_em_um_ano(year, team_id) for year in range(2012, 2026)]
    return seasons


def buscar_padroes_do_time(id_team, minsup=10):
    time = criar_sequencia_de_resultados_um_time(id_team)

    ps = PrefixSpan(time)
    padroes = ps.frequent(minsup=minsup)

    if len(padroes) > 0:
        padroes = sorted(padroes, key= lambda x: (-x[0], len(x[1])))
        for suporte, padrao in padroes:
            print(f"Suporte: {suporte}  \n\tPadrão: {padrao}")
            print("-"*200)

In [ ]:
# Cruzeiro
buscar_padroes_do_time(14, 10)

Suporte: 11  
	Padrão: [('W', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Baixa')]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Suporte: 11  
	Padrão: [('L', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Média')]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Suporte: 11  
	Padrão: [('W', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Baixa'), ('W', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Baixa')]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Suporte: 11  
	Padrão: [('W', 'Fez_Menos_2.5_Gols', 'Levou_Me

Análise para as sequências de partidas do Cruzeiro ao longo dos anos:

- Com suporte igual a 11, o Cruzeiro tende a ganhar dois jogos em sequência fazendo poucos gols quando sua odd de vitória é baixa

- Com suporte igual a 11, se a odd de vitória do primeiro jogo é baixa, ele tende a ganhar, mas se a segunda for média, ele tende a perder, podendo indicar um excesso de confiança

- Não há nenhuma sequência em que ganhou fazendo mais de 2.5 gols ou perdeu levando mais de 2.5 gols

- Com um suporte igual a 10, se os dois primeiros jogos forem derrotas com odds altas para vitória, o próximo tende a ser uma vitória ou empate com odd média para vitória

In [ ]:
# Atlético MG
buscar_padroes_do_time(3, 12)

Suporte: 14  
	Padrão: [('W', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Média')]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Suporte: 14  
	Padrão: [('L', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Alta')]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Suporte: 14  
	Padrão: [('D', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Alta')]
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
Suporte: 13  
	Padrão: [('W', 'Fez_Menos_2.5_Gols', 'Levou_Menos_2.5_Gols', 'Odd_Alta')]
----------------------------------------

Análise para as sequências de partidas do Atlético MG ao longo dos anos:

- Com suporte de 13, o Atlético tende a ganhar três partidas em sequência quando as odds para a sua vitória são baixas ou média
- Já com suporte 12 e odds de vitória baixas, ele tende a garantir a vitória em três partidas consecutivas
- Com um suporte de 12, tende a ganhar duas partidas seguidas mesmo quando as odds para sua vitória são médias ou altas, indicando bons resultados mesmo não sendo nada favorito
- A maior sequência de derrotas consecutivas ocorre com suporte igual a 12, mas somente ocorre quando as odds de vitória são altas